In [4]:
import math

def euclidean_distance(p, q):
    """Compute Euclidean distance between two m-dimensional points."""
    return math.sqrt(sum((pi - qi)**2 for pi, qi in zip(p, q)))

def distance_to_centers(point, centers):
    """Distance from a point to its nearest center."""
    return min(euclidean_distance(point, c) for c in centers)

def farthest_first_traversal(data, k):
    """
    Implement FarthestFirstTraversal(Data, k)
    First center = first point in data (as required by the challenge).
    """
    centers = [data[0]]  # first center is the first point

    while len(centers) < k:
        # Find the point farthest from its nearest center
        farthest_point = max(
            data,
            key=lambda p: distance_to_centers(p, centers)
        )
        centers.append(farthest_point)

    return centers


In [9]:
k = 3
m = 2

data_str = """0.0 0.0
5.0 5.0
0.0 5.0
1.0 1.0
2.0 2.0
3.0 3.0
1.0 2.0"""

data = [tuple(map(float, line.split())) for line in data_str.strip().split("\n")]

centers = farthest_first_traversal(data, k)

for c in centers:
    print(*c)

0.0 0.0
5.0 5.0
0.0 5.0


In [14]:
def squared_distance(p, q):
    """Squared Euclidean distance between two m-dimensional points."""
    return sum((pi - qi)**2 for pi, qi in zip(p, q))

def distortion(data, centers):
    """
    Compute the squared error distortion:
    average squared distance from each point to its nearest center.
    """
    total = 0
    for point in data:
        # distance to nearest center
        d = min(squared_distance(point, c) for c in centers)
        total += d
    return total / len(data)


# -----------------------------
# Example: parse input manually
# -----------------------------

k = 5
m = 2

centers_str = """8.2 9.6
4.8 9.1
9.3 0.4
6.3 8.6
0.1 6.0"""

data_str = """3.42 6.03
6.23 8.25
4.76 1.64
4.47 4.33
3.95 7.61
8.93 2.97
9.74 4.03
1.73 1.28
9.72 5.01
7.27 3.77"""

# Convert strings to lists of tuples
centers = [tuple(map(float, line.split())) for line in centers_str.split("\n")]
data = [tuple(map(float, line.split())) for line in data_str.split("\n")]

# Compute distortion
print(f"{distortion(data, centers):.3f}")

13.978


In [15]:
import math

def euclidean_distance(p, q):
    return math.sqrt(sum((pi - qi)**2 for pi, qi in zip(p, q)))

def compute_centroid(points, m):
    """Compute the mean of a list of m-dimensional points."""
    n = len(points)
    return [sum(p[i] for p in points) / n for i in range(m)]

def lloyd_kmeans(k, m, data):
    # Step 1: initialize centers as the first k points
    centers = [data[i][:] for i in range(k)]

    while True:
        # Step 2: assign each point to nearest center
        clusters = [[] for _ in range(k)]
        for point in data:
            distances = [euclidean_distance(point, c) for c in centers]
            nearest = distances.index(min(distances))
            clusters[nearest].append(point)

        # Step 3: update centers
        new_centers = []
        for cluster in clusters:
            new_centers.append(compute_centroid(cluster, m))

        # Step 4: check convergence
        if all(
            euclidean_distance(centers[i], new_centers[i]) < 1e-6
            for i in range(k)
        ):
            break

        centers = new_centers

    return centers


# --------------------------
# Example usage (matches page format)
# --------------------------
if __name__ == "__main__":
    # Sample Input from page
    k = 2
    m = 2
    data = [
        [1.3, 1.1],
        [1.3, 0.2],
        [0.6, 2.8],
        [3.0, 3.2],
        [1.2, 0.7],
        [1.4, 1.6],
        [1.2, 1.0],
        [1.2, 1.1],
        [0.6, 1.5],
        [1.8, 2.6],
        [1.2, 1.3],
        [1.2, 1.0],
        [0.0, 1.9],
    ]

    centers = lloyd_kmeans(k, m, data)

    # Output with 3 decimal places
    for c in centers:
        print(" ".join(f"{coord:.3f}" for coord in c))

1.800 2.867
1.060 1.140


In [3]:
def hierarchical_clustering_string_output(data):
    # 解析输入
    lines = data.strip().split("\n")
    n = int(lines[0])
    D = [list(map(float, line.split())) for line in lines[1:]]

    # 初始化簇：每个簇是一个列表（1-based index）
    clusters = {i: [i + 1] for i in range(n)}
    active = list(range(n))
    next_id = n

    # 输出结果（字符串行）
    output_lines = []

    # 深拷贝距离矩阵
    import copy
    Dmat = copy.deepcopy(D)

    while len(active) > 1:
        # 找到最小距离的两个簇
        min_dist = float("inf")
        ci, cj = None, None

        for i in active:
            for j in active:
                if i < j and Dmat[i][j] < min_dist:
                    min_dist = Dmat[i][j]
                    ci, cj = i, j

        # 合并簇
        new_cluster = clusters[ci] + clusters[cj]
        new_cluster_sorted = sorted(new_cluster)
        output_lines.append(" ".join(map(str, new_cluster_sorted)))

        # 注册新簇
        clusters[next_id] = new_cluster

        # 扩展距离矩阵
        size = len(Dmat)
        for row in Dmat:
            row.append(0.0)
        Dmat.append([0.0] * (size + 1))

        # 计算新簇与其他簇的距离（Davg）
        for c in active:
            if c != ci and c != cj:
                size_ci = len(clusters[ci])
                size_cj = len(clusters[cj])
                dist = (size_ci * Dmat[ci][c] + size_cj * Dmat[cj][c]) / (size_ci + size_cj)
                Dmat[next_id][c] = dist
                Dmat[c][next_id] = dist

        # 更新 active 列表
        active.remove(ci)
        active.remove(cj)
        active.append(next_id)

        next_id += 1

    return "\n".join(output_lines)

In [4]:
data = """7
0.00 0.74 0.85 0.54 0.83 0.92 0.89
0.74 0.00 1.59 1.35 1.20 1.48 1.55
0.85 1.59 0.00 0.63 1.13 0.69 0.73
0.54 1.35 0.63 0.00 0.66 0.43 0.88
0.83 1.20 1.13 0.66 0.00 0.72 0.55
0.92 1.48 0.69 0.43 0.72 0.00 0.80
0.89 1.55 0.73 0.88 0.55 0.80 0.00"""

print(hierarchical_clustering_string_output(data))

4 6
5 7
3 4 6
1 2
3 4 5 6 7
1 2 3 4 5 6 7


In [5]:
data = """20
0.00 0.28 1.08 1.24 0.49 0.56 1.59 0.79 1.14 1.73 0.65 1.40 1.36 0.77 1.28 0.92 1.50 0.98 1.21 1.20
0.28 0.00 0.73 1.21 0.88 0.66 1.50 1.17 1.55 1.52 1.28 1.39 1.19 0.60 1.20 0.89 1.36 0.83 0.96 0.84
1.08 0.73 0.00 0.55 0.90 0.85 1.03 1.19 1.26 0.88 1.49 1.01 1.09 0.84 1.15 1.18 1.18 1.06 1.08 1.25
1.24 1.21 0.55 0.00 0.80 1.37 0.67 0.76 0.71 0.69 1.49 0.75 1.19 0.68 1.41 1.16 0.70 1.55 1.17 1.51
0.49 0.88 0.90 0.80 0.00 0.37 0.98 0.64 0.91 1.62 0.67 1.11 1.04 0.70 1.38 0.57 0.89 0.73 1.25 1.89
0.56 0.66 0.85 1.37 0.37 0.00 1.19 1.09 1.20 1.58 0.66 1.25 1.09 0.66 0.84 0.41 1.10 0.37 1.18 1.40
1.59 1.50 1.03 0.67 0.98 1.19 0.00 0.80 0.58 0.93 1.11 0.89 0.52 0.97 0.63 0.89 0.20 0.98 1.00 1.14
0.79 1.17 1.19 0.76 0.64 1.09 0.80 0.00 0.28 1.53 0.71 0.44 0.79 1.16 1.26 1.34 1.06 1.14 1.81 1.51
1.14 1.55 1.26 0.71 0.91 1.20 0.58 0.28 0.00 1.10 0.55 0.70 1.14 1.28 0.72 1.28 1.00 1.49 1.66 1.28
1.73 1.52 0.88 0.69 1.62 1.58 0.93 1.53 1.10 0.00 1.39 0.90 1.16 1.13 0.91 1.19 0.85 1.39 0.55 0.64
0.65 1.28 1.49 1.49 0.67 0.66 1.11 0.71 0.55 1.39 0.00 1.22 1.13 1.42 0.68 0.99 1.30 0.98 1.24 1.15
1.40 1.39 1.01 0.75 1.11 1.25 0.89 0.44 0.70 0.90 1.22 0.00 0.54 1.07 1.32 1.38 0.90 0.86 1.57 1.26
1.36 1.19 1.09 1.19 1.04 1.09 0.52 0.79 1.14 1.16 1.13 0.54 0.00 1.10 1.05 1.12 0.48 0.39 0.97 1.02
0.77 0.60 0.84 0.68 0.70 0.66 0.97 1.16 1.28 1.13 1.42 1.07 1.10 0.00 1.13 0.66 0.59 0.85 1.04 1.27
1.28 1.20 1.15 1.41 1.38 0.84 0.63 1.26 0.72 0.91 0.68 1.32 1.05 1.13 0.00 0.94 0.92 1.00 0.99 0.52
0.92 0.89 1.18 1.16 0.57 0.41 0.89 1.34 1.28 1.19 0.99 1.38 1.12 0.66 0.94 0.00 0.62 0.51 0.64 1.20
1.50 1.36 1.18 0.70 0.89 1.10 0.20 1.06 1.00 0.85 1.30 0.90 0.48 0.59 0.92 0.62 0.00 0.75 0.71 1.17
0.98 0.83 1.06 1.55 0.73 0.37 0.98 1.14 1.49 1.39 0.98 0.86 0.39 0.85 1.00 0.51 0.75 0.00 0.90 1.11
1.21 0.96 1.08 1.17 1.25 1.18 1.00 1.81 1.66 0.55 1.24 1.57 0.97 1.04 0.99 0.64 0.71 0.90 0.00 0.56
1.20 0.84 1.25 1.51 1.89 1.40 1.14 1.51 1.28 0.64 1.15 1.26 1.02 1.27 0.52 1.20 1.17 1.11 0.56 0.00"""

print(hierarchical_clustering_string_output(data))

7 17
1 2
8 9
5 6
13 18
5 6 16
15 20
3 4
10 19
8 9 12
5 6 14 16
7 13 17 18
1 2 5 6 14 16
10 15 19 20
8 9 11 12
7 8 9 11 12 13 17 18
1 2 3 4 5 6 14 16
1 2 3 4 5 6 7 8 9 11 12 13 14 16 17 18
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20
